# The Anatomy of AI · Lecture 14 · Meaning as a Direction — the notebook

The lecture asked one question: a neuron is one direction among endless directions in a layer, so do the *other*
directions mean anything? Here you rebuild its answers with your own hands:

1. a neuron's weights are an **arrow**, and a layer is a bundle of arrows (Lectures 1 and 3, redrawn);
2. InceptionV1's layer **mixed4c** gives 512 numbers at each place: a point with 512 axes;
3. a **random arrow** among those axes, scored like a neuron, and what its reading is made of;
4. a panda that becomes a **gibbon** under a nudge of at most 2/255 per pixel, and why scattered nudges do nothing;
5. a **stripes direction** learned from texture photos, which pushes horses into zebras.

Each section names the lecture plate it follows. Wherever the lecture gives a number, a `check(...)` line compares
yours with it and prints ✓; if a check fails, the notebook stops.

**Runs on:** free Colab with a T4 GPU (Runtime → Change runtime type → T4), in about fifteen minutes. It downloads the
model (27 MB), the lecture's own copy of the ImageNet validation photos (two files of about 350 MB) and the Describable
Textures Dataset (two files, 630 MB in all). No keys.

## 0. Setup
Installs (pinned), the plot style, and `check()`.

In [ ]:
# Pinned installs. Free Colab already has torch and torchvision; a machine without them gets the pinned pair.
import importlib.util, subprocess, sys
def pip(*args):                                   # quiet: pip's output is shown only if an install fails
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1", "torchvision==0.20.1")
pip("numpy==2.0.2", "matplotlib==3.10.0", "pillow==11.1.0", "pyarrow==18.1.0", "requests==2.32.3")
pip("--no-deps", "torch-lucent==0.1.8")       # only its InceptionV1 port is used

In [ ]:
import io, json, math, os, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)      # library notices, not results
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", DEV)
T0 = time.time()

# The course's look: black ink on paper, no colour except real photos.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
    "lines.color": INK, "patch.edgecolor": INK, "axes.prop_cycle": plt.cycler(color=[INK, INK2]),
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=0):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(float(value) - expected) <= tol
        shown = f"{float(value):.6g}"
    else:
        ok = value == expected
        shown = repr(value)
    extra = f" ± {tol:g}" if isinstance(expected, (int, float)) and not isinstance(expected, bool) and tol else ""
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(expected)}{extra})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")

### The photos
The lecture used ImageNet's 50,000 validation photos, 256 pixels square, cut to the middle 224 × 224 (the size
InceptionV1 takes). We read the same files, byte for byte, and only the photos we ask for.

In [ ]:
# The photos: ImageNet's 50,000 validation photos, in the exact copy the course measured (a public 256-pixel copy,
# benjamin-paine/imagenet-1k-256x256 on Hugging Face: two parquet files of 25,000 photos, about 350 MB each, no key
# needed). Photo n here is photo n in the lecture. We read 100 photos at a time, only the ones we ask for.
import pyarrow.parquet as pq
import requests
from PIL import Image

SHARD = ("https://huggingface.co/datasets/benjamin-paine/imagenet-1k-256x256/resolve/main/data/"
         "validation-{:05d}-of-00002.parquet")
_files = {}


def _shard(s):
    path = f"val-{s}.parquet"
    if not os.path.exists(path):
        t = time.time()
        with requests.get(SHARD.format(s), stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
        os.rename(path + ".part", path)
        print(f"downloaded photo file {s} ({os.path.getsize(path) / 1e6:.0f} MB) in {time.time() - t:.0f} s")
    if s not in _files:
        _files[s] = pq.ParquetFile(path)
    return _files[s]


def rows(ids):
    """{n: (jpeg bytes, class number 0..999)} for the photos with these numbers."""
    groups = {}
    for n in ids:
        groups.setdefault((n // 25000, (n % 25000) // 100), []).append(int(n))
    out = {}
    for (s, g), ns in sorted(groups.items()):
        t = _shard(s).read_row_group(g, columns=["image", "label"])
        ims, labs = t.column("image").to_pylist(), t.column("label").to_pylist()
        for n in ns:
            out[n] = (ims[n % 100]["bytes"], int(labs[n % 100]))
    return out


def decode(b, size=224):
    """JPEG bytes (256 px square) -> the centre 224 x 224, as a 3 x 224 x 224 tensor in [0, 1]."""
    im = Image.open(io.BytesIO(b)).convert("RGB")
    w, h = im.size
    if (w, h) != (256, 256):
        s = 256 / min(w, h)
        im = im.resize((round(w * s), round(h * s)), Image.BICUBIC)
        w, h = im.size
    l, t = (w - size) // 2, (h - size) // 2
    im = im.crop((l, t, l + size, t + size))
    return torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0).permute(2, 0, 1)


def all_labels():
    """The class (0..999) of every one of the 50,000 photos, from the files' label column."""
    return np.concatenate([_shard(s).read(columns=["label"]).column("label").to_numpy() for s in (0, 1)])


def batches(ids, size=100):
    """Yield (ids, photos as an N x 3 x 224 x 224 tensor on the GPU), in order."""
    ids = [int(i) for i in ids]
    for k in range(0, len(ids), size):
        part = ids[k:k + size]
        R = rows(part)
        yield part, torch.stack([decode(R[n][0]) for n in part]).to(DEV)

### The network
InceptionV1 (GoogLeNet, 2014) reads a photo in layers. A *layer* is a stack of maps; each map is one *neuron*, slid
across the picture, so it has a value at every place. This lecture takes **gradients** (how much an answer changes
when an input changes), so the network is loaded with its true gradients; the cell explains the one setting that matters.

In [ ]:
# InceptionV1 (GoogLeNet, 2014), the network this lecture studies, through lucent's PyTorch port.
# One fix first: the port's local response normalization used the wrong constants (lucent pull request #51, Gorton 2024).
# We correct the installed file before importing it, as every lecture in the course does.
import importlib, importlib.util, pathlib
_spec = importlib.util.find_spec("lucent")
_f = pathlib.Path(_spec.origin).parent / "modelzoo" / "inceptionv1" / "InceptionV1.py"
_s = _f.read_text()
_old = "size=9, alpha=9.99999974738e-05, beta=0.5, k=1"
if _old in _s:
    _f.write_text(_s.replace(_old, "size=10, alpha=10 * 0.00009999999747378752, beta=0.5, k=2"))
for _p in (_f.parent / "__pycache__").glob("InceptionV1*.pyc"):
    _p.unlink()                                   # a stale cached copy would keep the old constants
importlib.invalidate_caches()

import contextlib
from lucent.modelzoo import inceptionv1
# Two settings matter because this lecture takes gradients:
#  - redirected_ReLU=False: lucent's default (True) is a feature-visualisation trick whose backward pass leaks gradient
#    through switched-off ReLUs; the lecture measured it at 0.51x the true slope. The forward pass is the same either way.
#  - TF32 off: on newer GPUs PyTorch would otherwise multiply with about 3 significant digits (a T4 has no TF32 anyway).
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
with contextlib.redirect_stderr(io.StringIO()):            # keep the download's percentage bar off the page
    net = inceptionv1(pretrained=True, redirected_ReLU=False).eval().to(DEV)
print("InceptionV1's weights (inception5h, 27 MB): ready, with true gradients")
for p in net.parameters():
    p.requires_grad_(False)

# Make sure the fix took: watch the normalization during one pass.
_M = importlib.import_module("lucent.modelzoo.inceptionv1.InceptionV1")
_seen, _orig = [], _M.F.local_response_norm
_M.F.local_response_norm = lambda x, size, alpha=1e-4, beta=0.75, k=1.0: (_seen.append((size, k)), _orig(x, size, alpha, beta, k))[1]
with torch.no_grad():
    net(torch.zeros(1, 3, 224, 224, device=DEV))
_M.F.local_response_norm = _orig
check("normalization size and k, every call", sorted(set(_seen)), [(10, 2)])


def prep(x01):
    """The port takes RGB in [-117, 138], as the original TensorFlow model did."""
    return x01 * 255.0 - 117.0


class Taps:
    """Look inside the network. record=[module names] keeps their outputs in taps[name];
    edit={name: fn} replaces a module's output with fn(output) on the way through."""
    def __init__(self, record=(), edit=None):
        mods = dict(net.named_modules())
        self.out, self.h = {}, []
        for name in record:
            self.h.append(mods[name].register_forward_hook(lambda m, i, o, n=name: self.out.__setitem__(n, o)))
        for name, fn in (edit or {}).items():
            self.h.append(mods[name].register_forward_hook(lambda m, i, o, fn=fn: fn(o)))

    def __getitem__(self, name):
        return self.out[name]

    def __enter__(self):
        return self

    def __exit__(self, *a):
        for h in self.h:
            h.remove()


# InceptionV1 has 1,008 outputs in its own order; PERM puts the 1,000 classes in the standard order (the lecture's
# perm.json, which also fixes two mislabelled outputs), and NAMES gives each class its name.
PERM = torch.tensor(json.loads("[449, 450, 442, 443, 444, 445, 446, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 494, 495, 496, 497, 498, 499, 500, 501, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 475, 476, 474, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 601, 602, 603, 604, 605, 606, 607, 608, 609, 610, 611, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 214, 215, 217, 216, 213, 218, 647, 648, 649, 650, 651, 652, 653, 654, 655, 656, 226, 613, 614, 615, 616, 617, 618, 619, 620, 612, 422, 423, 424, 425, 427, 426, 428, 429, 430, 438, 431, 432, 433, 434, 435, 436, 437, 439, 440, 441, 6, 22, 193, 14, 173, 99, 87, 69, 116, 198, 43, 89, 200, 98, 161, 132, 32, 180, 42, 18, 207, 181, 105, 77, 189, 20, 204, 63, 31, 66, 21, 25, 45, 170, 119, 210, 107, 126, 88, 145, 59, 160, 152, 72, 33, 91, 4, 27, 113, 123, 36, 156, 109, 158, 131, 26, 71, 56, 146, 144, 125, 176, 139, 134, 90, 2, 192, 154, 150, 94, 5, 19, 191, 196, 117, 141, 68, 93, 51, 208, 184, 97, 29, 171, 124, 128, 47, 64, 211, 112, 187, 114, 86, 115, 79, 130, 140, 110, 82, 17, 177, 149, 15, 3, 41, 127, 84, 143, 133, 60, 172, 179, 118, 168, 148, 70, 197, 50, 106, 49, 151, 46, 205, 102, 28, 58, 155, 136, 202, 34, 62, 1, 159, 67, 174, 55, 10, 95, 8, 11, 201, 85, 153, 30, 190, 76, 206, 61, 163, 103, 209, 74, 35, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 224, 631, 632, 633, 634, 635, 636, 637, 638, 639, 640, 641, 642, 643, 644, 645, 646, 225, 657, 658, 188, 129, 164, 157, 13, 53, 183, 195, 101, 39, 80, 147, 78, 120, 167, 108, 162, 165, 81, 52, 9, 65, 57, 12, 121, 186, 48, 23, 182, 40, 212, 44, 16, 178, 38, 83, 104, 96, 185, 122, 73, 135, 137, 138, 203, 54, 100, 175, 142, 166, 37, 111, 92, 199, 75, 194, 24, 7, 169, 447, 451, 448, 452, 453, 456, 457, 454, 455, 547, 853, 896, 223, 345, 246, 230, 232, 677, 265, 264, 522, 688, 845, 752, 540, 847, 704, 767, 233, 907, 967, 341, 718, 916, 307, 705, 683, 518, 905, 258, 807, 908, 296, 353, 785, 909, 884, 266, 733, 991, 849, 777, 811, 933, 941, 254, 985, 835, 533, 839, 689, 252, 940, 805, 300, 706, 779, 538, 817, 716, 872, 719, 949, 851, 820, 580, 833, 887, 707, 267, 673, 591, 539, 239, 377, 836, 576, 568, 891, 749, 563, 548, 890, 929, 701, 241, 987, 342, 914, 827, 721, 902, 382, 762, 303, 335, 301, 801, 690, 695, 370, 713, 797, 979, 761, 996, 674, 698, 583, 543, 708, 243, 268, 376, 347, 910, 881, 297, 545, 778, 898, 298, 670, 756, 856, 865, 720, 313, 550, 959, 966, 523, 529, 315, 821, 667, 579, 715, 253, 897, 972, 834, 336, 799, 1000, 662, 512, 346, 262, 316, 879, 660, 808, 796, 305, 235, 279, 919, 995, 356, 309, 784, 261, 712, 934, 299, 256, 348, 671, 757, 280, 971, 567, 955, 275, 792, 276, 236, 337, 911, 227, 684, 725, 703, 517, 581, 895, 249, 375, 840, 505, 551, 750, 573, 350, 344, 554, 369, 787, 696, 730, 562, 802, 924, 293, 525, 980, 659, 592, 748, 269, 961, 963, 294, 560, 770, 773, 582, 918, 892, 814, 228, 374, 988, 371, 687, 238, 546, 270, 244, 867, 973, 894, 508, 536, 697, 532, 818, 917, 782, 977, 763, 338, 339, 781, 984, 598, 922, 946, 302, 717, 509, 661, 866, 875, 920, 880, 271, 251, 871, 829, 290, 272, 764, 686, 869, 277, 824, 854, 691, 852, 260, 255, 728, 511, 599, 284, 588, 585, 595, 755, 915, 552, 699, 354, 351, 520, 513, 333, 870, 937, 295, 506, 921, 826, 564, 584, 504, 759, 685, 352, 877, 942, 994, 306, 527, 257, 679, 843, 732, 842, 850, 883, 783, 789, 575, 926, 722, 281, 596, 931, 901, 888, 841, 565, 245, 983, 372, 693, 964, 731, 381, 378, 857, 923, 285, 855, 314, 788, 838, 566, 373, 769, 556, 702, 542, 534, 572, 846, 939, 862, 976, 273, 860, 571, 863, 537, 927, 286, 570, 965, 668, 578, 694, 219, 541, 310, 663, 997, 876, 519, 760, 753, 586, 952, 751, 938, 355, 809, 521, 962, 221, 729, 510, 587, 379, 589, 559, 800, 709, 832, 950, 259, 380, 868, 747, 590, 776, 943, 549, 726, 557, 507, 288, 278, 960, 222, 986, 577, 925, 822, 858, 515, 234, 676, 237, 600, 775, 274, 593, 804, 263, 680, 340, 530, 998, 724, 528, 516, 514, 287, 957, 311, 692, 247, 794, 526, 574, 535, 810, 681, 828, 837, 945, 569, 561, 531, 304, 250, 978, 675, 786, 944, 970, 989, 903, 758, 555, 308, 780, 664, 710, 312, 594, 700, 282, 711, 289, 283, 766, 825, 291, 242, 597, 678, 882, 349, 765, 727, 544, 220, 292, 334, 666, 874, 990, 969, 558, 790, 682, 343, 936, 665, 524, 928, 317, 231, 906, 669, 958, 819, 795, 772, 502, 899, 912, 904, 935, 831, 503, 672, 951, 815, 723, 248, 240, 714, 553, 930, 791, 932, 861, 774, 803, 754, 813, 844, 771, 793, 974, 968, 873, 768, 975, 993, 885, 734, 736, 737, 738, 739, 740, 741, 742, 743, 744, 735, 745, 746, 318, 229, 319, 320, 321, 322, 323, 324, 325, 326, 816, 999, 953, 864, 806, 948, 830, 900, 798, 947, 859, 823, 361, 992, 359, 365, 368, 366, 363, 364, 367, 360, 362, 954, 848, 982, 330, 357, 358, 331, 327, 328, 332, 886, 913, 956, 893, 878, 812, 981, 329, 889]"), device=DEV)
from torchvision.models import GoogLeNet_Weights
NAMES = GoogLeNet_Weights.IMAGENET1K_V1.meta["categories"]


# The network's own 1,008 outputs; PERM[c] is where standard class c sits among them. NAME5H names any output.
_inv = {int(j): c for c, j in enumerate(PERM.tolist())}
def name5h(i):
    return NAMES[_inv[int(i)]] if int(i) in _inv else "(unused output)"
OUT = lambda c: int(PERM[c])                          # standard class number -> the network's output number

## 1. A neuron's weights are an arrow (Plates I–III)
Lecture 1's walk neuron reads a day as two numbers, warmth and rain. Its weights are 2.0 and −3.0 and its bias −0.5.
Draw the weights as a point on the same map as the days: from the corner, that is an **arrow**. The neuron *measures
how far a day reaches along its arrow* (the weighted sum, also called a dot product), then adds the bias.

In [ ]:
w, b = np.array([2.0, -3.0]), -0.5
sunny = np.array([0.8, 0.1])                         # Lecture 1's sunny day: warm, a little rain
reading = float(w @ sunny)
check("the sunny day, measured along the walk neuron's arrow", reading, 1.3, 1e-9)
check("plus the bias: the neuron's total", reading + b, 0.8, 1e-9)

fig, ax = plt.subplots(figsize=(4.6, 4.6))
u = w / np.linalg.norm(w)
ax.annotate("", xy=w / 3, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=INK, lw=2))
ax.text(*(w / 3 + [0.05, -0.05]), "the weights (2, −3), drawn a third as long", fontsize=9)
foot = (sunny @ u) * u                                # where the day lands on the arrow's line
ax.plot(*sunny, "o", color=INK); ax.text(sunny[0] + 0.03, sunny[1] + 0.03, "sunny day", fontsize=9)
ax.plot([sunny[0], foot[0]], [sunny[1], foot[1]], ls=":", color=INK)
t = np.linspace(-1, 1, 2)                             # the neuron's line: where the total is 0, at a right angle
perp = np.array([u[1], -u[0]])
ax.plot(*(-b / np.linalg.norm(w) * u[:, None] + perp[:, None] * t), color=INK2, ls="--")
ax.text(0.55, 0.62, "total = 0", color=INK2, fontsize=9)
ax.set_xlim(-0.2, 1.1); ax.set_ylim(-1.1, 1.0); ax.set_aspect("equal")
ax.set_xlabel("warmth"); ax.set_ylabel("rain"); ax.set_title("A neuron measures along its arrow", fontsize=11)
plt.show()

A **layer** is several neurons reading the same day: a bundle of arrows. Lecture 3's two "walls" read warmth only:
"too cold" has weights (−10, 0) and bias +3, "too hot" (10, 0) and bias −7, each through the gate (ReLU, which keeps
positive totals and turns negative ones to 0). Their two outputs are a new point for every day: one axis per neuron.
Then a combiner neuron (weights −2, −2, bias 1) reads *that* point, and says yes when its total is above 0.5.

In [ ]:
A_walls, c_walls = np.array([[-10.0, 0.0], [10.0, 0.0]]), np.array([3.0, -7.0])
v_comb, d_comb = np.array([-2.0, -2.0]), 1.0
diary = np.array([[0.6155, 0.7273, 0], [0.7775, 0.9159, 0], [0.0573, 0.4677, 0], [0.9168, 0.64, 0], [0.8768, 0.1364, 0], [0.4709, 0.2618, 1], [0.0423, 0.2337, 0], [0.7498, 0.18, 0], [0.7793, 0.1604, 0], [0.6104, 0.1491, 1], [0.0317, 0.8491, 0], [0.2269, 0.2326, 0], [0.9535, 0.8501, 0], [0.5369, 0.6672, 0], [0.2225, 0.9145, 0], [0.8701, 0.3109, 0], [0.3695, 0.186, 1], [0.167, 0.0912, 0], [0.0332, 0.6673, 0], [0.7994, 0.4819, 0], [0.9466, 0.0515, 0], [0.7348, 0.8242, 0], [0.047, 0.7705, 0], [0.0385, 0.0739, 0], [0.2001, 0.9279, 0], [0.2147, 0.7404, 0], [0.9039, 0.9155, 0], [0.3537, 0.3635, 1], [0.5232, 0.7591, 0], [0.1316, 0.7335, 0], [0.5424, 0.3237, 1], [0.4115, 0.2531, 1], [0.4583, 0.4265, 1], [0.4803, 0.3036, 1], [0.4096, 0.1677, 1], [0.4994, 0.3482, 1], [0.623, 0.0694, 1], [0.4733, 0.2427, 1], [0.4674, 0.3875, 1], [0.3814, 0.0639, 1], [0.6068, 0.1191, 1], [0.5425, 0.349, 1]])                             # Lecture 3's 42 days: warmth, rain, good walk (1) or not (0)
days, good = diary[:, :2], diary[:, 2] == 1
h = np.maximum(0, days @ A_walls.T + c_walls)          # the layer's answer: a point (cold, hot) for each day
check("days on the 'too cold' axis only", int(((h[:, 0] > 0) & (h[:, 1] == 0)).sum()), 12)
check("days on the 'too hot' axis only", int(((h[:, 1] > 0) & (h[:, 0] == 0)).sum()), 11)
check("days at the corner (both walls silent)", int((h == 0).all(1).sum()), 19)
check("days off both axes (both walls firing)", int((h > 0).all(1).sum()), 0)
yes = h @ v_comb + d_comb > 0.5
check("the combiner says yes to", int(yes.sum()), 19)
check("days it gets right, of 42", int((yes == good).sum()), 39)

Every day lands on an axis or at the corner, because the gate keeps only one wall awake at a time. The combiner's
weights are one more arrow, now on the walls' map. Any arrow there is a *mix* of the two neurons.

## 2. 512 axes, and an arrow chosen at random (Plates IV–V)
InceptionV1's layer **mixed4c** looks at a photo in 14 × 14 places and gives 512 numbers at each: one per neuron. So
one place is one point in a space with 512 axes. A neuron is one axis. A **direction** is any arrow in that space, any
mix of the 512. The lecture drew 50 arrows at random (each number from a bell curve, then scaled to length 1), from a
fixed seed; we draw the same ones. Its **random arrow #3** reads this photo of daisies (validation photo 4,919) best.

In [ ]:
rng = np.random.default_rng(1400 + 512)               # the lecture's seed for mixed4c's directions
NEURONS50 = np.sort(rng.choice(512, 50, replace=False))  # 50 neurons it sampled (drawn first, to keep the sequence)
RANDOM50 = rng.standard_normal((512, 50))
RANDOM50 /= np.linalg.norm(RANDOM50, axis=0)           # 50 random arrows of length 1
v3 = torch.tensor(RANDOM50[:, 3], dtype=torch.float32, device=DEV)

DAISY = 4919
daisy_jpg, daisy_label = rows([DAISY])[DAISY]
daisy = decode(daisy_jpg)
with torch.no_grad(), Taps(["mixed4c"]) as T:
    net(prep(daisy[None].to(DEV)))
A4c = T["mixed4c"][0]                                 # 512 x 14 x 14
check("mixed4c: neurons x height x width", tuple(A4c.shape), (512, 14, 14))
check("the photo's class", NAMES[daisy_label], "daisy")
reading_map = torch.einsum("c,chw->hw", v3, A4c)      # the arrow's reading at every place
pos = int(reading_map.flatten().argmax())
check("random arrow #3's best place (row x 14 + column)", pos, 77)
a = A4c.flatten(1)[:, pos]                            # the 512 numbers at that place
# (the lecture counted from its stored, rounded numbers, so a value of 0.0001 there reads as 0: allow two)
check("of the 512 neurons there, 0 (silenced by the gate)", int((a == 0).sum()), 300, 2)
check("random arrow #3's reading there", float(v3 @ a), 546.008, 0.5)

fig, axs = plt.subplots(1, 2, figsize=(9.5, 3.6), gridspec_kw=dict(width_ratios=[1, 2.2]))
axs[0].imshow(daisy.permute(1, 2, 0).numpy()); axs[0].axis("off")
r_, c_ = divmod(pos, 14)
axs[0].add_patch(plt.Rectangle((c_ * 16, r_ * 16), 16, 16, fill=False, ec=PAPER, lw=2))
axs[0].set_title(f"photo {DAISY:,} and the best place", fontsize=10)
axs[1].bar(range(512), a.cpu().numpy(), width=1.0, color=INK)
axs[1].set_xlabel("neuron of mixed4c"); axs[1].set_title("its 512 numbers: one point with 512 axes", fontsize=10)
plt.show()

**Scoring an arrow** is Lecture 6's recipe: read every photo along it at every place, keep the best place, and rank
the photos. The lecture ranked all 50,000; we rank the first 5,000, which is enough to check the order: the lecture's
global top 50 restricted to these photos must come out first, in the same order. We score neuron **4c:490** (a neuron
is just the arrow along one axis) and random arrow #3, plus three more of each kind for an exercise later.

In [ ]:
GUESS = [int(NEURONS50[k]) for k in (5, 20, 35)] + [7, 21, 44]   # three sampled neurons, three random arrows
cols = [np.eye(512)[490], RANDOM50[:, 3]] + [np.eye(512)[n] for n in GUESS[:3]] + [RANDOM50[:, k] for k in GUESS[3:]]
VD = torch.tensor(np.stack(cols), dtype=torch.float32, device=DEV)   # 8 arrows x 512
N = 5000
best, where = np.zeros((N, len(cols)), np.float32), np.zeros((N, len(cols)), np.int64)
t0 = time.time()
for ids, X in batches(range(N)):
    with torch.no_grad(), Taps(["mixed4c"]) as T:
        net(prep(X))
    m, at = torch.einsum("dc,bchw->bdhw", VD, T["mixed4c"]).flatten(2).max(-1)
    best[ids], where[ids] = m.cpu().numpy(), at.cpu().numpy()
print(f"read {N:,} photos along 8 arrows in {time.time() - t0:.0f} s")
top = {k: [int(i) for i in np.argsort(-best[:, k])[:9]] for k in range(len(cols))}
check("4c:490's five best of the first 5,000", top[0][:5], [970, 289, 3570, 1649, 338])
check("random #3's five best of the first 5,000", top[1][:5], [4919, 2210, 1064, 1925, 4513])


def crop(n, k, size=128):
    """The 128 x 128 square of photo n around arrow k's best place."""
    im = decode(rows([n])[n][0])
    r_, c_ = divmod(int(where[n, k]), 14)
    cy, cx = (r_ + 0.5) * 16, (c_ + 0.5) * 16
    y0, x0 = int(min(max(cy - size / 2, 0), 224 - size)), int(min(max(cx - size / 2, 0), 224 - size))
    return im[:, y0:y0 + size, x0:x0 + size].permute(1, 2, 0).numpy()


def strip(k, title):
    fig, axs = plt.subplots(1, 9, figsize=(11, 1.6))
    for ax, n in zip(axs, top[k]):
        ax.imshow(crop(n, k)); ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel(f"{n}", fontsize=7)
    fig.suptitle(title, fontsize=10, y=1.02)
    plt.show()


strip(0, "neuron 4c:490: its nine best of the first 5,000")
strip(1, "random arrow #3: its nine best of the first 5,000")

Both look like they *mean* something: 4c:490 likes glowing digital displays, and the random arrow likes daisies.
In 270 panels each rated by two blind raters, the lecture could not tell neurons from random arrows (Plates VII–VIII, at 4c and 4e: neurons 3.05, random arrows 2.97, out of 5). That test needs raters, so it is an exercise here: you rate (Your turn 6).

## 3. Inside a random arrow (Plate X)
A reading is a sum: for each neuron, its weight on the arrow times its output, added over all 512. Sort the 512
pieces, largest first. Where does the daisy reading come from?

In [ ]:
pieces = (v3 * a).cpu().numpy()
srt = np.sort(pieces)[::-1]
check("the 512 pieces add up to the reading", float(pieces.sum()), 546.008, 0.5)
check("share of the reading from the 10 largest pieces", float(srt[:10].sum() / pieces.sum()), 0.65, 0.003)
print("the three largest pieces come from neurons", [int(i) for i in np.argsort(-pieces)[:3]])

fig, ax = plt.subplots(figsize=(9, 2.8))
ax.bar(range(512), np.where(srt > 0, srt, 0), width=1.0, color=INK)
ax.bar(range(512), np.where(srt < 0, srt, 0), width=1.0, color=PAPER, edgecolor=INK, lw=0.3)
ax.axvspan(-0.5, 9.5, color=INK2, alpha=0.15)
ax.set_xlabel("the 512 pieces, largest first"); ax.set_title("ten neurons carry most of a random arrow's reading", fontsize=10)
plt.show()

Is that one lucky arrow? The lecture did the same for all 50 random arrows, at each one's nine best photos of the
50,000 (their numbers and places are written below, from the lecture's log), and averaged.

In [ ]:
TOP9 = {"0": [[2210, 48], [29439, 156], [6919, 119], [45475, 90], [2760, 62], [9817, 108], [4919, 63], [36492, 104], [43695, 104]], "1": [[5615, 63], [36273, 50], [39805, 51], [33702, 63], [27848, 119], [19769, 88], [11506, 37], [32773, 36], [21171, 75]], "2": [[26831, 57], [41543, 64], [4049, 106], [19287, 38], [21637, 7], [16591, 48], [14139, 100], [40901, 23], [26456, 63]], "3": [[4919, 77], [13438, 77], [48958, 78], [43695, 118], [41193, 88], [34228, 128], [2210, 62], [6983, 106], [23820, 119]], "4": [[6316, 49], [46144, 90], [44325, 29], [30348, 36], [46160, 17], [11552, 22], [8411, 156], [39620, 63], [44101, 20]], "5": [[46850, 48], [5574, 76], [7001, 36], [5582, 75], [4803, 64], [35940, 73], [24283, 80], [7631, 54], [33377, 23]], "6": [[49826, 18], [43560, 43], [48032, 193], [18109, 184], [10345, 141], [32411, 28], [43044, 185], [6867, 28], [1085, 42]], "7": [[24329, 117], [20867, 36], [35332, 37], [40908, 60], [2297, 18], [34615, 43], [13970, 127], [19602, 91], [14922, 20]], "8": [[45087, 72], [30814, 93], [35718, 86], [38155, 140], [9714, 146], [750, 90], [24265, 35], [48646, 179], [76, 106]], "9": [[44868, 9], [7119, 88], [32753, 47], [49863, 9], [11700, 28], [32553, 98], [21207, 57], [43891, 85], [708, 84]], "10": [[5360, 132], [31315, 21], [28309, 30], [47097, 18], [22666, 99], [39576, 58], [1756, 159], [7803, 16], [10636, 86]], "11": [[13215, 119], [33895, 40], [38206, 20], [6904, 133], [24778, 48], [9016, 76], [45604, 34], [41208, 87], [26109, 170]], "12": [[12160, 60], [44236, 89], [36439, 121], [42074, 64], [34276, 76], [33772, 100], [18366, 73], [37790, 106], [13157, 77]], "13": [[20845, 14], [17039, 154], [30176, 146], [9149, 2], [23312, 141], [36420, 110], [5837, 11], [30154, 25], [42157, 11]], "14": [[11528, 173], [45304, 37], [21715, 140], [23435, 57], [28533, 95], [9556, 49], [6291, 31], [41792, 180], [21894, 20]], "15": [[42564, 74], [29283, 94], [12576, 61], [25536, 72], [6666, 92], [31529, 103], [49682, 169], [408, 90], [29496, 72]], "16": [[21837, 155], [38543, 76], [49161, 182], [12248, 58], [8346, 7], [21060, 151], [19065, 115], [46255, 19], [44332, 49]], "17": [[25770, 131], [16649, 135], [11894, 76], [34268, 65], [11256, 76], [28025, 118], [4075, 49], [40422, 100], [34228, 100]], "18": [[29594, 101], [22295, 161], [17890, 36], [23927, 107], [40586, 60], [17880, 36], [46694, 140], [22742, 127], [4919, 63]], "19": [[43872, 91], [11749, 61], [10807, 137], [40095, 115], [2890, 104], [23927, 107], [45087, 81], [46598, 88], [38106, 37]], "20": [[7917, 21], [32096, 63], [36515, 12], [13582, 46], [7634, 118], [42714, 67], [34723, 36], [4099, 78], [2492, 118]], "21": [[45817, 34], [29854, 117], [36850, 104], [12790, 104], [1925, 51], [1444, 88], [32528, 47], [22188, 48], [440, 77]], "22": [[15092, 34], [39580, 100], [21417, 86], [31747, 48], [12700, 48], [21342, 37], [16957, 25], [32482, 33], [5683, 30]], "23": [[2210, 49], [32317, 107], [34228, 114], [23737, 134], [10804, 75], [15963, 100], [16494, 90], [41193, 89], [42448, 29]], "24": [[30348, 104], [2311, 166], [47216, 95], [9333, 48], [33292, 105], [11151, 39], [36110, 62], [32943, 131], [7918, 90]], "25": [[43872, 91], [23146, 34], [43893, 36], [38818, 29], [40366, 42], [3769, 47], [10807, 137], [8411, 131], [45089, 60]], "26": [[40797, 36], [29592, 3], [47264, 6], [30610, 76], [12079, 62], [16625, 6], [16957, 25], [27400, 171], [10743, 46]], "27": [[39509, 102], [4099, 107], [25644, 90], [6034, 90], [27668, 38], [24636, 75], [29380, 162], [10330, 47], [38293, 75]], "28": [[13132, 63], [6936, 58], [2748, 61], [37702, 64], [37675, 36], [27481, 75], [45445, 93], [32645, 37], [22433, 62]], "29": [[10871, 89], [10747, 20], [26109, 156], [43893, 36], [33269, 48], [21291, 88], [6638, 146], [8289, 39], [9656, 40]], "30": [[14578, 195], [31181, 167], [42520, 114], [22254, 195], [40540, 43], [36645, 140], [2563, 183], [30033, 192], [40650, 116]], "31": [[38200, 88], [45945, 76], [9842, 134], [32528, 49], [29854, 115], [45817, 33], [48499, 24], [38660, 73], [22821, 61]], "32": [[38070, 34], [27107, 64], [48958, 89], [45892, 35], [22091, 64], [28131, 48], [16082, 64], [46147, 60], [32830, 35]], "33": [[5683, 30], [7803, 30], [41256, 20], [31623, 104], [49680, 20], [42693, 106], [37537, 7], [37860, 34], [45304, 37]], "34": [[38106, 37], [4641, 166], [21729, 95], [23436, 90], [39056, 39], [19188, 61], [36850, 132], [19172, 49], [29360, 62]], "35": [[32818, 110], [173, 45], [30923, 79], [24283, 66], [20131, 76], [15328, 74], [40586, 74], [12229, 82], [37080, 22]], "36": [[33419, 62], [1149, 90], [30051, 102], [19454, 92], [38200, 61], [22507, 90], [18751, 49], [18780, 91], [41632, 77]], "37": [[13453, 81], [1493, 92], [21171, 61], [6167, 46], [10248, 67], [48760, 88], [2707, 36], [45817, 35], [4099, 93]], "38": [[10583, 47], [7525, 65], [33641, 75], [15328, 73], [2062, 50], [5469, 141], [20675, 36], [14922, 33], [874, 35]], "39": [[7037, 85], [17748, 86], [44663, 154], [6159, 92], [31, 152], [3111, 90], [2890, 90], [32112, 136], [6477, 107]], "40": [[15328, 86], [29360, 63], [22382, 151], [27640, 35], [46850, 62], [47706, 58], [40074, 92], [13893, 162], [31315, 63]], "41": [[7772, 59], [16950, 65], [21171, 75], [48626, 76], [23538, 36], [49199, 58], [33819, 39], [12607, 145], [5316, 77]], "42": [[15092, 34], [21417, 86], [16381, 71], [6326, 101], [47063, 63], [13911, 104], [24326, 20], [19107, 43], [37251, 35]], "43": [[5418, 36], [16186, 7], [18569, 19], [2314, 61], [38249, 45], [30072, 30], [32701, 60], [21358, 107], [26110, 33]], "44": [[11403, 104], [19769, 88], [21417, 100], [49199, 58], [10330, 47], [24532, 52], [44545, 116], [26110, 74], [49513, 62]], "45": [[49339, 57], [19886, 81], [10466, 95], [23175, 54], [43805, 58], [13598, 174], [4942, 93], [16355, 34], [12404, 91]], "46": [[8590, 48], [23759, 92], [25771, 104], [427, 90], [28242, 73], [43267, 46], [25825, 89], [43805, 30], [24767, 49]], "47": [[3684, 109], [41223, 108], [45087, 81], [28701, 119], [4227, 137], [12301, 109], [32848, 102], [8136, 104], [26594, 106]], "48": [[3521, 20], [39490, 20], [7506, 21], [32087, 21], [39438, 90], [45108, 75], [37251, 34], [27484, 31], [44552, 77]], "49": [[19769, 88], [10330, 47], [24428, 49], [49199, 72], [5615, 63], [45150, 133], [4099, 108], [44552, 121], [38293, 75]]}                                          # random arrow k -> its nine best [photo, place] of 50,000


def anatomy(arrows, best9):
    """Average share of the reading from the 1 and the 10 largest pieces, over each arrow's nine best photos."""
    need = sorted({g for k in best9 for g, _ in best9[k]})
    acts = {}
    for ids, X in batches(need):
        with torch.no_grad(), Taps(["mixed4c"]) as T:
            net(prep(X))
        for n, A_ in zip(ids, T["mixed4c"].flatten(2).cpu().numpy()):
            acts[n] = A_
    s1, s10 = [], []
    for k, v in arrows.items():
        for g, p in best9[k]:
            c = v * acts[g][:, p]
            srt_ = np.sort(c)[::-1]
            s1.append(srt_[0] / c.sum()); s10.append(srt_[:10].sum() / c.sum())
    return float(np.mean(s1)), float(np.mean(s10))


share1, share10 = anatomy({str(k): RANDOM50[:, k] for k in range(50)}, TOP9)
check("random arrows: share from their largest piece, on average", share1, 0.1764, 0.005)
check("random arrows: share from their 10 largest pieces, on average", share10, 0.9403, 0.005)

On its best photos, a random arrow's reading is carried by about ten neurons firing hard together: it borrows its
meaning from a few neurons. That is one reason a random arrow can *look* meaningful.

## 4. A panda becomes a gibbon (Plates XII–XV)
A photo is numbers too: 224 × 224 places × 3 colours = 150,528 numbers between 0 and 1. So a photo is a point with
150,528 axes, and the network's answer is a function of that point. Our panda is validation photo 42,189.

In [ ]:
PANDA = 42189
I_PANDA, I_GIBBON = OUT(NAMES.index("giant panda")), OUT(NAMES.index("gibbon"))
x0 = decode(rows([PANDA])[PANDA][0])[None].to(DEV)
check("numbers in one photo", x0.numel(), 150528)


def logits(x):
    """The network's 1,008 scores before the softmax (logits)."""
    with Taps(["softmax2_pre_activation_matmul"]) as T:
        net(prep(x))
    return T["softmax2_pre_activation_matmul"]


def verdict(x):
    with torch.no_grad():
        z = logits(x)[0]
    p = z.softmax(0)
    k = int(p.argmax())
    return name5h(k), float(p[k]), float(z[I_PANDA]), float(z[I_GIBBON]), float(p[I_PANDA]), float(p[I_GIBBON])


v0 = verdict(x0)
print("clean:", v0[:2])
check("the clean photo's answer", v0[0], "giant panda")
check("P(giant panda)", v0[1], 1.0, 1e-4)
check("panda's score (logit)", v0[2], 33.156, 0.05)
check("gibbon's score (logit)", v0[3], 11.936, 0.05)

**The gradient** of the gap *gibbon score − panda score* says, for each of the 150,528 numbers, which way to move it to
raise the gap. Moving every number one step of 2/255 (the smallest change an 8-bit image file can hold, twice) along the
sign of its gradient is Goodfellow's one-step nudge. Then the lecture's ten small steps of 0.5/255, re-aimed each time,
never leaving ±2/255 of the photo.

In [ ]:
q8 = lambda x: (x.clamp(0, 1) * 255).round() / 255     # what an 8-bit image file can hold


def gap_grad(x):
    x = x.clone().requires_grad_(True)
    z = logits(x)[0]
    g, = torch.autograd.grad(z[I_GIBBON] - z[I_PANDA], x)
    return g


one = verdict(q8(x0 + (2 / 255) * gap_grad(x0).sign()))
print("one aimed step of 2/255:", one[:2])
check("one step: the answer", one[0], "Great Pyrenees")
check("one step: its probability", one[1], 0.6064, 0.02)


def ten_steps(x0, box=2 / 255, step=0.5 / 255, iters=10):
    x = x0.clone()
    for _ in range(iters):
        x = torch.min(torch.max(x + step * gap_grad(x).sign(), x0 - box), x0 + box).clamp(0, 1)
    return q8(x).detach()


xa = ten_steps(x0)
va = verdict(xa)
print("ten small steps:", va[:2])
check("ten steps: the answer", va[0], "gibbon")
check("ten steps: P(gibbon)", va[5], 1.0, 1e-3)
# (ten re-aimed steps amplify the last-digit differences between GPUs, so the scores may differ by a point or two)
check("ten steps: gibbon's score", va[3], 52.416, 2.5)
check("ten steps: panda's score", va[2], 2.244, 2.5)
delta = xa - x0
d255 = (delta * 255).round().int()
check("the largest change to any number, in 255ths", int(d255.abs().max()), 2)
hist = {int(k): int(c) for k, c in zip(*torch.unique(d255, return_counts=True))}
print("how many numbers moved by -2, -1, 0, +1, +2 (in 255ths):", hist)
# the split between 1/255 and 2/255 also depends on the GPU's rounding; the share of numbers moved does not
LECTURE_HIST = {"-2": 20263, "-1": 30655, "0": 37052, "1": 30445, "2": 32113}
print("the lecture's:", LECTURE_HIST)
check("share of the 150,528 numbers moved at all", 1 - hist.get(0, 0) / delta.numel(), 0.7539, 0.01)

**The control.** Take exactly the same 150,528 changes and shuffle them to random places: same sizes, same counts,
no aim. Then sweep the strength t from 0 to 2 for both.

In [ ]:
torch.manual_seed(5)
shuffled = delta.flatten()[torch.randperm(delta.numel(), device=DEV)].view_as(delta)
vs = verdict((x0 + shuffled).clamp(0, 1))
print("the same changes, shuffled:", vs[:2])
check("shuffled: the answer", vs[0], "giant panda")
check("shuffled: P(giant panda)", vs[4], 1.0, 1e-3)

ts = np.round(np.arange(0, 2.0001, 0.05), 2)
curves = {}
with torch.no_grad():
    for nm_, dd in (("aimed", delta), ("shuffled", shuffled)):
        X = (x0 + torch.tensor(ts, dtype=torch.float32, device=DEV).view(-1, 1, 1, 1) * dd).clamp(0, 1)
        P = logits(X).softmax(1)
        curves[nm_] = (P[:, I_PANDA].cpu().numpy(), P[:, I_GIBBON].cpu().numpy())
flip = float(ts[np.argmax(curves["aimed"][1] > curves["aimed"][0])])
check("strength at which the aimed nudge flips panda to gibbon", flip, 0.35, 0.05)
fig, ax = plt.subplots(figsize=(6.5, 3))
ax.plot(ts, curves["aimed"][1], color=INK, lw=2, label="aimed: P(gibbon)")
ax.plot(ts, curves["aimed"][0], color=INK, lw=1, ls="--", label="aimed: P(panda)")
ax.plot(ts, curves["shuffled"][0], color=INK2, lw=2, label="shuffled: P(panda)")
ax.set_xlabel("strength t (1 = the ten-step nudge)"); ax.set_ylabel("probability"); ax.legend(fontsize=8)
plt.show()

g0 = gap_grad(x0)
aimed_push, shuffled_push = float((g0 * delta).sum()), float((g0 * shuffled).sum())
print(f"first-order push on the gap: aimed {aimed_push:+.1f}, shuffled {shuffled_push:+.1f}")
check("aimed: sum over 150,528 numbers of (slope x change)", aimed_push, 48.54, 2.4)
check("shuffled push is under 5% of the aimed one", abs(shuffled_push) < 0.05 * aimed_push, True)

Aimed, 150,528 tiny pushes all add up; scattered, they cancel. Is it one lucky panda? The lecture ran the same ten
steps on all 50 pandas in the validation set.

In [ ]:
labels50k = all_labels()
pandas = np.where(labels50k == NAMES.index("giant panda"))[0]
check("pandas in the validation set", len(pandas), 50)
after = []
for ids, X in batches(pandas, size=10):
    for k in range(len(ids)):
        after.append(verdict(ten_steps(X[k:k + 1]))[0])
print({nm_: after.count(nm_) for nm_ in sorted(set(after), key=after.count, reverse=True)})
check("pandas that become gibbons", after.count("gibbon"), 46, 2)

**How straight is the network?** Goodfellow et al. (2014) explained the nudge by treating the network as one straight
line near the photo. The lecture measured that in float64 (no rounding at all): move every number by ε along the sign of
its slope, and compare the real change in the gap with the straight-line guess ε × Σ|slope|.

In [ ]:
import copy
net64 = copy.deepcopy(net).double()
mods64 = dict(net64.named_modules())
x64 = x0.double()


def gap64(x):
    out = {}
    h = mods64["softmax2_pre_activation_matmul"].register_forward_hook(lambda m, i, o: out.__setitem__("z", o))
    net64(prep(x))
    h.remove()
    return out["z"][0, I_GIBBON] - out["z"][0, I_PANDA]


xb = x64.clone().requires_grad_(True)
G0 = gap64(xb)
g64, = torch.autograd.grad(G0, xb)
d = g64.sign()
movable = ~(((x64 <= 0) & (d < 0)) | ((x64 >= 1) & (d > 0)))      # numbers already at 0 or 1 cannot move further
ratios = {}
with torch.no_grad():
    for e255 in (1e-4, 0.1):
        e = e255 / 255
        pred = float(e * (g64.abs() * movable).sum())
        up = float(gap64((x64 + e * d).clamp(0, 1)) - G0)
        dn = float(G0 - gap64((x64 - e * d).clamp(0, 1)))
        ratios[e255] = (up / pred, dn / pred)
        print(f"nudge {e255:g}/255: real change / straight-line guess = {up / pred:.3f} (up), {dn / pred:.3f} (down)")
check("straightness at 0.0001/255, up", ratios[1e-4][0], 0.9937, 0.005)
check("straightness at 0.0001/255, down", ratios[1e-4][1], 0.9996, 0.005)
check("straightness at 0.1/255, up", ratios[0.1][0], 0.8245, 0.01)
check("straightness at 0.1/255, down", ratios[0.1][1], 0.7355, 0.01)
del net64

Straight only very near the photo: at 0.0001/255 the guess is right to within 1%, at 0.1/255 the real change is only
three-quarters to four-fifths of it. The nudge works anyway, because 150,528 small aimed pushes add up to far more than is needed.

## 5. A direction we can name (Plates XVI–XVIII)
Kim et al. (2018) asked the question the other way round: pick an idea we have a word for, *stripes*, and find its
direction. Show mixed4c 120 photos of striped textures and 100 ordinary photos, and fit a straight cut between the two
piles (a **logistic regression**). The arrow across the cut is the **concept activation vector**: the stripes direction.
The textures come from the Describable Textures Dataset (Cimpoi et al. 2014), 47 textures × 120 photos.

In [ ]:
# The Describable Textures Dataset, from the ungated mirror the lecture used (tanganke/dtd: train 80 + test 40 per texture).
DTD = "https://huggingface.co/datasets/tanganke/dtd/resolve/main/data/{}-00000-of-00001.parquet"
TEXTURES = ("banded blotchy braided bubbly bumpy chequered cobwebbed cracked crosshatched crystalline dotted fibrous "
            "flecked freckled frilly gauzy grid grooved honeycombed interlaced knitted lacelike lined marbled matted "
            "meshed paisley perforated pitted pleated polka-dotted porous potholed scaly smeared spiralled sprinkled "
            "stained stratified striped studded swirly veined waffled woven wrinkled zigzagged").split()
CONCEPTS = ["striped", "zigzagged", "dotted"]
texture_jpgs = {c: [] for c in CONCEPTS}
for split in ("train", "test"):
    path = f"dtd-{split}.parquet"
    if not os.path.exists(path):
        with requests.get(DTD.format(split), stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(path, "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
    labs = pq.read_table(path, columns=["label"]).column("label").to_pylist()
    keep_ = [k for k, lab in enumerate(labs) if TEXTURES[lab] in texture_jpgs]
    ims = pq.read_table(path, columns=["image"]).column("image").take(keep_).to_pylist()
    for k, im in zip(keep_, ims):
        texture_jpgs[TEXTURES[labs[k]]].append(im["bytes"])
print("textures:", {c: len(v) for c, v in texture_jpgs.items()})
check("striped texture photos", len(texture_jpgs["striped"]), 120)


def texture(b, size=224):
    """As the lecture stored them (JPEG, quality 95), then the middle square at 224 x 224."""
    buf = io.BytesIO()
    Image.open(io.BytesIO(b)).convert("RGB").save(buf, "JPEG", quality=95)
    im = Image.open(buf).convert("RGB")
    w_, h_ = im.size
    s = min(w_, h_)
    im = im.crop(((w_ - s) // 2, (h_ - s) // 2, (w_ - s) // 2 + s, (h_ - s) // 2 + s)).resize((size, size), Image.BILINEAR)
    return torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0).permute(2, 0, 1)


def acts4c(X):
    out = []
    for k in range(0, len(X), 60):
        with torch.no_grad(), Taps(["mixed4c"]) as T:
            net(prep(X[k:k + 60].to(DEV)))
        out.append(T["mixed4c"].flatten(1).cpu())
    return torch.cat(out)


concept_acts = {c: acts4c(torch.stack([texture(b) for b in texture_jpgs[c]])) for c in CONCEPTS}
fig, axs = plt.subplots(1, 9, figsize=(11, 1.4))
for ax, k in zip(axs, range(0, 120, 14)):
    ax.imshow(texture(texture_jpgs["striped"][k]).permute(1, 2, 0).numpy()); ax.axis("off")
fig.suptitle("some of the 120 striped textures", fontsize=10, y=1.05)
plt.show()

In [ ]:
# 20 piles of 100 ordinary photos, drawn as the lecture drew them (no zebras, tigers, dalmatians or horses).
CLASSES = {"zebra": 340, "tiger": 292, "dalmatian": 251, "sorrel": 339}
rng = np.random.default_rng(14)
pool = np.where(~np.isin(labels50k, list(CLASSES.values())))[0]
rand_sets = [np.sort(rng.choice(pool, 100, replace=False)) for _ in range(21)]   # set 0 is the lecture's null pile
need = sorted(set(np.concatenate(rand_sets[1:]).tolist()))
t0 = time.time()
rand_store = {}
for ids, X in batches(need):
    with torch.no_grad(), Taps(["mixed4c"]) as T:
        net(prep(X))
    for n, A_ in zip(ids, T["mixed4c"].flatten(1).cpu()):
        rand_store[n] = A_
print(f"{len(need):,} ordinary photos read in {time.time() - t0:.0f} s")


def cav(pos, neg, seed, alpha=0.01, iters=200):
    """The lecture's cut: L2-regularised logistic regression (alpha 0.01, as Kim et al.'s code), features divided by
    their mean spread, solved with LBFGS. Returns the unit arrow pointing toward the concept."""
    Xc = torch.cat([pos, neg]).to(DEV)
    y = torch.cat([torch.ones(len(pos)), torch.zeros(len(neg))]).to(DEV)
    Xc = Xc / (Xc.std(0).mean() + 1e-8)
    torch.manual_seed(seed)
    w_ = torch.zeros(Xc.shape[1], device=DEV, requires_grad=True)
    b_ = torch.zeros(1, device=DEV, requires_grad=True)
    opt = torch.optim.LBFGS([w_, b_], lr=1, max_iter=iters, line_search_fn="strong_wolfe")

    def closure():
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(Xc @ w_ + b_, y) + 0.5 * alpha * (w_ * w_).sum()
        loss.backward()
        return loss
    opt.step(closure)
    return (w_ / w_.norm()).detach()


t0 = time.time()
cavs = {c: [cav(concept_acts[c], torch.stack([rand_store[int(n)] for n in rand_sets[i]]), i) for i in range(1, 21)]
        for c in CONCEPTS}
print(f"60 cuts fitted in {time.time() - t0:.0f} s")

**TCAV.** For each of the 50 zebra photos, take the slope of the zebra score with respect to mixed4c's numbers, and
measure it along the stripes arrow: positive means "more stripes, more zebra". The **TCAV score** is the share of zebras
with a positive slope, averaged over the 20 cuts. Zigzags and dots are the comparison.

In [ ]:
zebras = np.where(labels50k == CLASSES["zebra"])[0]
keep = {}
h = dict(net.named_modules())["mixed4c"].register_forward_hook(lambda m, i, o: (o.retain_grad(), keep.__setitem__("a", o))[0])
Rz = rows(zebras.tolist())
Z = torch.stack([decode(Rz[n][0]) for n in zebras.tolist()]).to(DEV).requires_grad_(True)
logits(Z)[:, OUT(CLASSES["zebra"])].sum().backward()
h.remove()
Gz = keep["a"].grad.flatten(1)                          # 50 zebras x 100,352 slopes
tcav = {c: float(np.mean([float((Gz @ v > 0).float().mean()) for v in cavs[c]])) for c in CONCEPTS}
print("TCAV scores for 'zebra' at mixed4c:", {c: round(s, 3) for c, s in tcav.items()})
check("TCAV: stripes", tcav["striped"], 1.0, 0.02)
check("TCAV: zigzags", tcav["zigzagged"], 0.676, 0.04)
check("TCAV: dots", tcav["dotted"], 0.169, 0.04)
mean_cav = {c: (lambda m_: m_ / m_.norm())(torch.stack(cavs[c]).mean(0)) for c in CONCEPTS}
pos_slopes = {c: int((Gz @ mean_cav[c] > 0).sum()) for c in CONCEPTS}
check("zebras sloping up along the (average) stripes arrow, of 50", pos_slopes["striped"], 50)
check("... along the zigzags arrow", pos_slopes["zigzagged"], 34, 3)
check("... along the dots arrow", pos_slopes["dotted"], 9, 3)

**Push the horses.** A slope is a hint; pushing is a test. Add the stripes arrow inside mixed4c to each of the 50 horse
photos (class "sorrel"), scaled to the photo's own activation length: a′ = a + t × ‖a‖ × v, with t = 1. Compare a random
arrow pushed just as hard.

In [ ]:
horses = np.where(labels50k == CLASSES["sorrel"])[0]
Rh = rows(horses.tolist())
H = torch.stack([decode(Rh[n][0]) for n in horses.tolist()]).to(DEV)
rnd = np.random.default_rng(3).standard_normal((512, 14, 14))
directions = {"striped": mean_cav["striped"], "zigzagged": mean_cav["zigzagged"], "dotted": mean_cav["dotted"],
              "random": torch.tensor(rnd / np.linalg.norm(rnd), dtype=torch.float32, device=DEV).flatten()}


def push(v, t=1.0):
    def edit(o):
        n_ = o.flatten(1).norm(dim=1).view(-1, 1, 1, 1)
        return o + t * n_ * v.view(1, *o.shape[1:])
    with torch.no_grad(), Taps(["softmax2_pre_activation_matmul"], edit={"mixed4c": edit}) as T:
        net(prep(H))
    tops = T["softmax2_pre_activation_matmul"].argmax(1)
    return [name5h(i) for i in tops]


pushed = {nm_: push(v) for nm_, v in directions.items()}
for nm_, labs in pushed.items():
    print(f"{nm_:10s}", sorted({l: labs.count(l) for l in set(labs)}.items(), key=lambda kv: -kv[1])[:4])
check("unpushed horses called 'sorrel'", push(directions["random"], 0.0).count("sorrel"), 48)
check("pushed along stripes: zebras", pushed["striped"].count("zebra"), 31, 3)
check("pushed along stripes: tigers", pushed["striped"].count("tiger"), 10, 3)
check("pushed along a random arrow: zebras", pushed["random"].count("zebra"), 0)

fig, axs = plt.subplots(1, 8, figsize=(11, 1.7))
for ax, k in zip(axs, range(0, 50, 6)):
    ax.imshow(H[k].permute(1, 2, 0).cpu().numpy()); ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel(pushed["striped"][k], fontsize=8)
fig.suptitle("horses, and what the network calls them after the stripes push", fontsize=10, y=1.03)
plt.show()

Pushed along stripes, horses become zebras and tigers; pushed along a random arrow just as hard, they stay horses. Some
directions mean the same thing to us and to the network.

**What we did not redo here:** the blind test's ratings and the slow turn from the neuron axes to random axes
(Plates VII–IX) need the full 50,000-photo census and raters; the TCAV rank of stripes among all 47 textures (Plate
XVII's note) needs every texture's cuts. Their numbers are in the lecture's `FINDINGS.md`.

## Your turn
Each exercise has a one-line hint; worked solutions are at the end.

1. **A pair of neurons.** The lecture also scored 50 *pairs*: arrows that mix exactly two neurons equally. Using the
   same anatomy as Section 3, how much of a pair's reading comes from its largest piece, on average? *Hint:* the pairs
   are drawn right after the random arrows from the same `rng`; `PAIR9` below holds their best photos.
2. **A smaller nudge.** Make the one aimed step 1/255 instead of 2/255. What does the panda become? *Hint:* reuse
   `gap_grad` and `q8`.
3. **Shuffle again.** Shuffle the ten-step nudge with three other seeds. Does any shuffle fool the network?
4. **Other textures.** Push the horses along zigzags and along dots (already in `pushed`). Any zebras?
5. **Why aimed wins, on paper.** Draw n random slopes g from a bell curve. An aimed nudge adds Σ|g|; a random-sign
   nudge adds Σ±g, whose spread is ‖g‖. How does the ratio grow with n, and what is it at n = 150,528?
6. **Neuron or random?** Six strips below are three neurons and three random arrows, in a shuffled order. Guess which
   is which before you read the solution.

In [ ]:
PAIR9 = {"231+84": [[31758, 149], [6326, 87], [33691, 82], [4795, 78], [31014, 65], [4178, 89], [2273, 40], [35432, 116], [18963, 102]], "428+144": [[25435, 104], [22219, 146], [5037, 90], [9412, 89], [12759, 90], [1867, 104], [27367, 90], [31596, 45], [11018, 102]], "454+182": [[40908, 88], [31874, 64], [33789, 105], [31916, 91], [5138, 48], [19168, 89], [29931, 47], [36536, 90], [30393, 77]], "486+297": [[35801, 81], [29584, 108], [30470, 142], [41403, 74], [39121, 135], [10219, 58], [22124, 22], [46553, 101], [8314, 60]], "462+420": [[23609, 94], [30276, 46], [48980, 108], [32928, 51], [4394, 32], [11167, 118], [12866, 94], [2844, 103], [24193, 35]], "463+205": [[49199, 58], [27640, 50], [24428, 49], [40855, 71], [23120, 74], [21488, 43], [2460, 43], [41048, 58], [27063, 74]], "362+308": [[14784, 129], [36019, 161], [43267, 46], [21347, 48], [3191, 117], [44297, 87], [12641, 36], [44825, 33], [16059, 105]], "131+417": [[46885, 35], [37104, 62], [4349, 48], [41937, 48], [46319, 76], [26870, 90], [40553, 47], [47045, 130], [35748, 131]], "108+250": [[11201, 3], [16249, 99], [1445, 112], [16630, 71], [17283, 127], [33810, 101], [47733, 0], [21198, 42], [12235, 178]], "388+184": [[45581, 61], [13555, 20], [36518, 95], [33668, 47], [45944, 90], [48703, 39], [25337, 94], [23146, 75], [9918, 90]], "350+369": [[27425, 61], [26776, 93], [30724, 122], [19571, 91], [20492, 128], [28328, 114], [44601, 66], [41980, 136], [21575, 121]], "186+90": [[4919, 63], [49417, 67], [6983, 106], [39297, 90], [49493, 89], [46543, 77], [13993, 105], [10142, 101], [47869, 63]], "433+352": [[14912, 131], [2634, 148], [28410, 178], [15328, 74], [11623, 52], [46973, 149], [42868, 150], [43215, 142], [29775, 131]], "406+284": [[26405, 124], [29898, 68], [10466, 136], [21879, 66], [41175, 69], [19240, 54], [29860, 40], [33163, 92], [16537, 37]], "19+333": [[44297, 30], [6696, 20], [21894, 20], [38826, 23], [659, 18], [22142, 36], [35705, 20], [3714, 25], [25906, 45]], "463+469": [[14903, 18], [24496, 64], [8957, 162], [43852, 128], [95, 76], [30675, 90], [19498, 161], [2210, 91], [49055, 46]], "240+223": [[12277, 49], [28369, 116], [18765, 15], [44545, 130], [43908, 113], [21813, 128], [10337, 15], [7518, 121], [16063, 15]], "347+87": [[4352, 154], [22230, 0], [37801, 141], [6424, 60], [21273, 17], [27328, 19], [34749, 190], [6659, 126], [22251, 5]], "334+439": [[48203, 133], [13659, 75], [40439, 103], [40402, 51], [37408, 24], [2371, 85], [26776, 93], [32433, 92], [28475, 115]], "282+303": [[6248, 122], [19832, 75], [10747, 34], [17912, 48], [47045, 130], [7804, 51], [23126, 101], [16742, 24], [6904, 146]], "478+60": [[8572, 50], [45313, 143], [1182, 70], [23248, 47], [26175, 5], [11345, 112], [20484, 168], [23153, 3], [39008, 112]], "50+275": [[38513, 174], [8075, 32], [24311, 76], [20101, 49], [15962, 103], [49020, 20], [40917, 107], [48020, 146], [34356, 49]], "349+473": [[6983, 106], [48958, 78], [2210, 62], [4919, 77], [34228, 142], [37871, 65], [9187, 48], [36492, 118], [18751, 35]], "34+209": [[16494, 77], [43270, 102], [24341, 90], [24151, 50], [12466, 62], [48381, 90], [25160, 35], [32611, 85], [27816, 25]], "365+145": [[32340, 100], [12250, 16], [19188, 150], [17645, 34], [15328, 60], [43872, 91], [21565, 133], [40439, 90], [24053, 45]], "145+15": [[43872, 91], [17850, 106], [32321, 56], [12250, 16], [46721, 73], [34437, 90], [6652, 29], [7245, 99], [33400, 42]], "416+75": [[12900, 28], [47653, 112], [18125, 59], [40426, 59], [18839, 56], [46729, 29], [48514, 42], [49856, 73], [31820, 15]], "347+19": [[40736, 173], [2009, 150], [7300, 145], [29844, 15], [32443, 133], [6050, 146], [24598, 77], [24277, 104], [10251, 132]], "312+375": [[38484, 1], [16788, 141], [31618, 141], [13970, 141], [9617, 69], [20259, 95], [30553, 12], [3830, 12], [11710, 127]], "86+416": [[20117, 50], [1871, 14], [16997, 50], [40586, 74], [37080, 22], [29798, 63], [34573, 84], [45979, 64], [21488, 43]], "282+391": [[36850, 102], [29925, 106], [18881, 102], [47147, 103], [32701, 76], [17289, 61], [41307, 61], [27518, 91], [1567, 107]], "450+274": [[41544, 129], [45850, 118], [8062, 106], [23534, 100], [4075, 61], [9526, 148], [25762, 100], [31415, 144], [46333, 103]], "311+223": [[49881, 36], [30692, 130], [9538, 103], [2634, 119], [45107, 155], [11745, 136], [40950, 131], [9931, 143], [13019, 29]], "220+394": [[16919, 156], [38098, 142], [41577, 75], [38655, 58], [43447, 156], [40500, 170], [34967, 158], [45976, 141], [35651, 157]], "62+69": [[38164, 70], [29719, 156], [1508, 105], [3396, 36], [48296, 8], [41632, 103], [43861, 175], [48414, 29], [24397, 39]], "289+44": [[49176, 115], [20828, 110], [148, 174], [26125, 103], [14384, 131], [21813, 128], [25468, 104], [18652, 160], [16336, 124]], "149+387": [[19127, 36], [6943, 67], [13702, 92], [15250, 80], [23737, 35], [37998, 77], [13104, 65], [39459, 92], [6058, 33]], "56+444": [[48498, 76], [34223, 20], [39509, 60], [3710, 48], [30827, 34], [35472, 48], [1095, 50], [6833, 75], [25990, 35]], "395+492": [[33428, 118], [25435, 91], [12759, 76], [2210, 49], [33037, 62], [28243, 90], [8530, 64], [47225, 64], [16888, 104]], "324+128": [[49176, 129], [17650, 145], [19045, 117], [745, 149], [22550, 145], [8038, 144], [40074, 120], [40458, 160], [23717, 148]], "273+31": [[48834, 29], [11906, 19], [20492, 34], [25818, 29], [44525, 50], [40402, 58], [9026, 47], [24264, 18], [47485, 14]], "271+328": [[38106, 37], [7350, 39], [4803, 50], [26339, 33], [47290, 26], [32290, 43], [46445, 58], [36737, 17], [34989, 21]], "63+431": [[20745, 175], [15145, 35], [30398, 148], [48535, 45], [45355, 103], [9333, 29], [5922, 6], [42860, 39], [1807, 160]], "212+52": [[49417, 67], [24930, 155], [28502, 176], [38520, 163], [39264, 58], [42573, 109], [3210, 154], [36892, 47], [3233, 175]], "498+356": [[14372, 178], [36986, 130], [23087, 162], [27743, 17], [20040, 120], [21192, 21], [32826, 113], [21097, 104], [49162, 110]], "345+63": [[12422, 118], [21035, 160], [19778, 58], [32065, 104], [20745, 175], [15145, 35], [21660, 144], [48535, 45], [21916, 116]], "280+68": [[49263, 90], [23175, 26], [31798, 1], [4964, 29], [6937, 46], [40689, 71], [47253, 163], [38826, 123], [4955, 98]], "235+410": [[33665, 24], [48420, 86], [42288, 121], [22644, 66], [13320, 81], [19388, 44], [23309, 66], [26900, 46], [17438, 31]], "488+251": [[7891, 72], [11745, 136], [35940, 45], [17399, 123], [16318, 81], [16147, 88], [26, 86], [49199, 15], [5574, 49]], "395+260": [[19188, 61], [10531, 76], [22667, 121], [24374, 175], [21284, 90], [34480, 77], [43526, 73], [27366, 122], [25435, 91]]}                                        # pair "i+j" -> its nine best [photo, place] of 50,000
# TODO 1: rebuild the 50 pair arrows (keep drawing from rng after RANDOM50), then use anatomy(...)

In [ ]:
# TODO 2: one aimed step of 1/255

In [ ]:
# TODO 3: three more shuffles of `delta`

In [ ]:
# TODO 4: count zebras in pushed['zigzagged'] and pushed['dotted']

In [ ]:
# TODO 5: aimed vs random sums for n = 10 ... 150,528

In [ ]:
order6 = [2, 5, 3, 7, 4, 6]                            # columns of `best`: 2-4 are neurons, 5-7 random arrows, shuffled
for k_, col in enumerate(order6):
    strip(col, f"strip {k_ + 1}")
# TODO 6: write down your six guesses (neuron or random) before reading the solution

## Solutions

In [ ]:
# 1. The pairs: drawn from the same rng, right after the 50 random arrows.
rng_ = np.random.default_rng(1400 + 512)
rng_.choice(512, 50, replace=False); rng_.standard_normal((512, 50))
pairs = {}
for _ in range(50):
    i, j = rng_.choice(512, 2, replace=False)
    v = np.zeros(512); v[i] = v[j] = 1 / np.sqrt(2)
    pairs[f"{i}+{j}"] = v
check("the pairs drawn match the lecture's", sorted(pairs) == sorted(PAIR9), True)
p1, p10 = anatomy(pairs, PAIR9)
check("pairs: share from the largest piece", p1, 0.7826, 0.005)
check("pairs: share from the 10 largest pieces", p10, 1.0, 1e-6)
print("A pair is mostly one neuron at a time: on each best photo, one of its two neurons does about 80% of the work.")

In [ ]:
# 2. One step of 1/255 is not enough to reach a gibbon, but it already leaves the panda.
v1 = verdict(q8(x0 + (1 / 255) * gap_grad(x0).sign()))
print("one aimed step of 1/255:", v1[:2], " P(panda) =", round(v1[4], 3))
check("1/255: the answer", v1[0], "Great Pyrenees")
check("1/255: its probability", v1[1], 0.3432, 0.03)

In [ ]:
# 3. Three more shuffles: none fools it.
for seed in (1, 2, 3):
    torch.manual_seed(seed)
    sh = delta.flatten()[torch.randperm(delta.numel(), device=DEV)].view_as(delta)
    v_ = verdict((x0 + sh).clamp(0, 1))
    check(f"shuffle with seed {seed}: still a panda", v_[0], "giant panda")

In [ ]:
# 4. Zigzags and dots make no zebras: the push works only along the direction that means stripes.
check("pushed along zigzags: zebras", pushed["zigzagged"].count("zebra"), 0)
check("pushed along dots: zebras", pushed["dotted"].count("zebra"), 0)

In [ ]:
# 5. Aimed grows like n, random like the square root of n: their ratio grows like sqrt(2n / pi) for bell-curve slopes.
rng5 = np.random.default_rng(0)
ns = np.unique(np.logspace(1, np.log10(150528), 12).astype(int))
ratio = []
for n in ns:
    g = rng5.standard_normal(n)
    ratio.append(np.abs(g).sum() / np.linalg.norm(g))
fig, ax = plt.subplots(figsize=(5, 3))
ax.loglog(ns, ratio, "o-", color=INK, label="aimed ÷ spread of random")
ax.loglog(ns, np.sqrt(2 * ns / np.pi), color=INK2, ls="--", label="√(2n/π)")
ax.set_xlabel("n, numbers nudged"); ax.legend(fontsize=8); plt.show()
check("ratio at n = 150,528, against sqrt(2n/pi)", ratio[-1] / np.sqrt(2 * 150528 / np.pi), 1.0, 0.01)

In [ ]:
# 6. The answer: strips 1-6 were
truth = ["neuron" if col in (2, 3, 4) else "random" for col in order6]
names6 = [f"4c:{GUESS[col - 2]}" if col in (2, 3, 4) else f"random #{GUESS[col - 2]}" for col in order6]
for k_, (t_, n_) in enumerate(zip(truth, names6)):
    print(f"strip {k_ + 1}: {t_:7s} ({n_})")
print("The lecture's raters, on 100 of each, scored neurons 3.05 and random arrows 2.97 out of 5: no measurable gap.")

In [ ]:
print(f"notebook finished in {(time.time() - T0) / 60:.1f} min")